# iterable과 iterator — 반복할 대상과 읽는 진행 위치

[전체 목차](../../README.md) · [도구 목차](../README.md) · [이 장 목차](README.md)

**이 장의 목표:** 같은 리스트는 다시 읽히는데, 같은 map은 왜 두 번째에 비어 있는지 설명한다.
용어를 외우기 전에 아래 두 실행 결과를 비교하세요. 각 코드 셀은 독립 실행할 수 있습니다.

## 1. 개념과 모범 코드 — 다시 쓸 결과는 list에 저장

순회할 수 있는 대상을 **iterable(이터러블)**이라고 합니다. 리스트, 문자열, 튜플, range 등이 그렇습니다.
다음 값을 어디까지 읽었는지 진행 위치를 갖고 하나씩 제공하는 객체는 **iterator(이터레이터)**입니다.
map, zip, enumerate, reversed의 결과가 iterator입니다.

**이건 코테에서 자주 쓰니 챙겨라: map 결과를 여러 번 쓸 거라면 list로 한 번 모아 둔다.**


In [ ]:
numbers = list(map(int, ['10', '20', '30']))
print(sum(numbers))  # 예상 출력: 60
print(sum(numbers))  # 예상 출력: 60
print(numbers)  # 예상 출력: [10, 20, 30]


반대로 map 객체 자체를 저장해 두면 다음과 같습니다.


In [ ]:
converted = map(int, ['10', '20', '30'])
print(sum(converted))  # 예상 출력: 60
print(sum(converted))  # 예상 출력: 0


두 번째 sum이 0인 이유는 **남은 원소가 없기 때문**입니다. sum은 빈 입력의 합을 0으로 반환합니다.
계산 결과가 지워졌다거나 원본 숫자들이 0으로 바뀐 것이 아닙니다.

| 시점 | 같은 map 객체에 남은 변환 대상 | sum 결과 |
| --- | --- | --- |
| 생성 직후 | `'10'`, `'20'`, `'30'` | 아직 합산 안 함 |
| 첫 sum 이후 | 없음 | 60 |
| 두 번째 sum 이후 | 없음 | 0 |

## 2. iter로 읽기 시작하고, next로 하나씩 읽는다

`iter(대상)`은 대상을 읽을 iterator를 얻습니다. 리스트를 전달하면 처음부터 읽을 새 iterator가 생깁니다.
`next(iterator)`는 다음 값 하나를 꺼냅니다. `next(iterator, 기본값)`은 다 읽은 경우 기본값을 줍니다.


In [ ]:
values = [10, 20, 30]
reader = iter(values)
print(next(reader))  # 예상 출력: 10
print(next(reader))  # 예상 출력: 20
print(list(reader))  # 예상 출력: [30]
print(next(reader, '끝'))  # 예상 출력: 끝
print(values)  # 예상 출력: [10, 20, 30]


| 실행한 표현 | 반환값 | reader가 다음에 읽을 값 |
| --- | --- | --- |
| `iter(values)` | 리스트 iterator | 10 |
| 첫 `next(reader)` | 10 | 20 |
| 두 번째 `next(reader)` | 20 | 30 |
| `list(reader)` | `[30]` | 없음 |
| `next(reader, '끝')` | `'끝'` | 없음 |

**소비했다 = 이 iterator의 진행 위치가 앞으로 갔다**는 뜻입니다.
원본 리스트에서 원소를 삭제했다는 뜻이 아닙니다. 리스트 자체는 여전히 `[10, 20, 30]`입니다.

## 3. for문은 이 일을 자동으로 해준다

보통 코테 풀이에서 iter와 next를 직접 적을 필요는 없습니다. for문이 대상을 읽고 끝에서 멈춥니다.


In [ ]:
values = [10, 20, 30]
first = []
second = []
for value in values:
    first.append(value)
for value in values:
    second.append(value)
print(first)  # 예상 출력: [10, 20, 30]
print(second)  # 예상 출력: [10, 20, 30]


두 for문은 리스트에서 각자 읽을 iterator를 얻으므로 처음부터 읽습니다.
그러나 **이미 만들어 둔 같은 iterator**를 다시 주면 남은 곳부터 읽습니다.


In [ ]:
reader = iter([10, 20, 30])
first = []
second = []
for value in reader:
    first.append(value)
for value in reader:
    second.append(value)
print(first)  # 예상 출력: [10, 20, 30]
print(second)  # 예상 출력: []


iterator도 for문에 넣을 수 있으므로 iterable입니다. **모든 iterator는 iterable이지만, 모든 iterable이 iterator인 것은 아닙니다.**
리스트는 iterable이며 그 자체에 next를 사용할 수는 없습니다.

## 4. iter(iterator)는 되감기가 아니다

리스트로 새 iterator를 만들 수는 있지만, 같은 iterator를 iter에 넣어도 처음으로 돌아가지 않습니다.


In [ ]:
values = [10, 20]
reader = iter(values)
print(next(reader))  # 예상 출력: 10
same_reader = iter(reader)
print(same_reader is reader)  # 예상 출력: True
print(list(same_reader))  # 예상 출력: [20]
print(list(iter(values)))  # 예상 출력: [10, 20]


`is`는 두 변수가 같은 객체를 가리키는지 확인합니다. `same_reader`는 새로 시작한 읽기 도구가 아니라 reader 그 자체입니다.
다시 읽으려면 원본 리스트에서 새 iterator를 얻거나, 처음부터 결과를 list에 저장해 둡니다.

## 5. 무엇을 꺼내는지와 다시 읽을 수 있는지

| 대상 | 직접 순회할 때 나오는 값 | 같은 객체를 끝까지 읽고 다시 순회 |
| --- | --- | --- |
| `[10, 20]` | 정수 원소 | 다시 처음부터 |
| `'ab'` | 한 글자 문자열 | 다시 처음부터 |
| `range(2)` | 0, 1 | 다시 처음부터 |
| `{'a': 10, 'b': 20}` | 키 `'a'`, `'b'` | 다시 처음부터 |
| `{10, 20}` | 집합 원소, 순서에 의존하지 않음 | 다시 순회 가능 |
| `iter([10, 20])` | 정수 원소 | 남은 값 없음 |
| `map(int, ['1', '2'])` | 변환한 정수 | 남은 값 없음 |
| `enumerate(['a', 'b'])` | `(번호, 값)` 튜플 | 남은 값 없음 |
| `zip([1], [2])` | `(1, 2)` 튜플 | 남은 값 없음 |
| `reversed([1, 2])` | 2, 1 | 남은 값 없음 |

딕셔너리의 직접 순회는 값이 아니라 **키**입니다. 값만 필요하면 values, 키와 값은 items를 씁니다.


In [ ]:
scores = {'민수': 80, '지수': 90}
print(list(scores))  # 예상 출력: ['민수', '지수']
print(list(scores.values()))  # 예상 출력: [80, 90]
print(list(scores.items()))  # 예상 출력: [('민수', 80), ('지수', 90)]


딕셔너리 도구의 자세한 설명은 [dict 문서](../05-집합과-딕셔너리/dict.md)에 있습니다.
정수 `3`은 순회할 수 없습니다. 3번 반복하려는 뜻이면 `range(3)`을 사용합니다.

## 6. 입력이 iterator면 원본 진행 위치도 움직인다

map은 원본 리스트를 수정하지 않지만, 입력 iterator에서 값을 읽으면 그 iterator가 소비됩니다.


In [ ]:
source = iter(['10', '20', '30'])
converted = map(int, source)
print(next(converted))  # 예상 출력: 10
print(list(source))  # 예상 출력: ['20', '30']
print(list(converted))  # 예상 출력: []


첫 next에서 map은 source의 `'10'`을 읽어 10으로 바꿉니다.
그 뒤 `list(source)`가 나머지 `'20'`, `'30'`을 직접 읽어 버렸습니다.
그래서 converted가 추가로 읽을 입력도 없습니다. **서로 연결된 iterator를 번갈아 소비하면 결과가 달라질 수 있습니다.**

## 7. 문법·예외·비용과 가져갈 것

| 도구 | 입력 → 반환 | 경계 조건 |
| --- | --- | --- |
| `iter(대상)` | iterable → iterator | 빈 리스트도 iterator를 만들 수 있음 |
| `next(reader)` | iterator → 다음 원소 자체 | 끝이면 `StopIteration` |
| `next(reader, 기본값)` | iterator → 다음 원소 또는 기본값 | 끝이어도 예외 없이 기본값 |
| `list(reader)` | iterator → 남은 원소의 list | 끝이면 `[]` |


In [ ]:
empty = iter([])
print(list(empty))  # 예상 출력: []
print(next(empty, None))  # 예상 출력: None


`iter(3)`은 반복할 수 없는 입력이므로 `TypeError`입니다.
`next([1, 2])`는 iterable이지만 iterator는 아닌 리스트를 전달해 `TypeError`입니다.
직접 next를 쓸 때는 다음 장의 기본값 처리를 함께 봅니다.

리스트 iterator 생성과 다음 원소 접근은 O(1)입니다. n개 원소를 저장하면 시간·공간 O(n)이 듭니다.
일반 iterator의 다음 값을 계산하는 비용은 내부 작업에 따라 달라집니다. map이 비싼 함수를 실행하면 그 비용도 포함됩니다.

1. iterable은 순회 가능한 대상이고, iterator는 소비되는 진행 상태를 갖는다.
2. list로 **저장된 결과**와 map 같은 **읽는 객체**를 구분한다.
3. 다시 읽을 결과는 저장한다. 합계 한 번만 구한다면 바로 sum으로 읽는다.

다음: [next — 값 하나만 꺼내기](next.ipynb) · 다시 보기: [map](map.ipynb), [zip/reversed](zip-reversed.ipynb)

문법 확인: [Python 공식 문서 — iterator](https://docs.python.org/3.9/library/stdtypes.html#iterator-types), [iter](https://docs.python.org/3.9/library/functions.html#iter)
